**Анализ эффективности кампаний: **
1. Сравните эффективность различных кампаний с точки зрения генерации
лидов и коэффициента конверсии.
2. Оцените эффективность различных маркетинговых источников Source в
генерировании качественных лидов.

In [5]:
import pandas as pd

spend_df = pd.read_excel("Spend_f_clean.xlsx")

display(spend_df.head())

spend_df.info()

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19862 entries, 0 to 19861
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         19862 non-null  datetime64[ns]
 1   Source       19862 non-null  object        
 2   Campaign     14785 non-null  object        
 3   Impressions  19862 non-null  int64         
 4   Spend        19862 non-null  float64       
 5   Clicks       19862 non-null  int64         
 6   AdGroup      13951 non-null  object        
 7   Ad           13951 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.2+ MB


In [2]:
cols = ["Campaign", "Impressions", "Spend", "Clicks"]

fill_rate = (
    spend_df[cols].count() / len(spend_df) * 100
).round(1)

fill_rate = fill_rate.rename("Заполненность (%)")
print(fill_rate)




Campaign        74.4
Impressions    100.0
Spend          100.0
Clicks         100.0
Name: Заполненность (%), dtype: float64


In [3]:
spend_df["Campaign"].nunique()


51

Хочу вывести таблицу всех кампаний с расчетными полями. Ее можно сортировать по нужному полю и смотреть на топ по показателям.

In [4]:
# --- Агрегация по кампаниям ---
campaign_stats = (
    spend_df.groupby("Campaign", dropna=True)[["Impressions", "Spend", "Clicks"]]
    .sum()
    .reset_index()
)

# --- Добавляем метрики ---
campaign_stats["Conversion Rate (%)"] = (
    campaign_stats["Clicks"] / campaign_stats["Impressions"] * 100
).round(2)

campaign_stats["Cost per Click"] = (
    campaign_stats["Spend"] / campaign_stats["Clicks"]
).round(2)

# --- Сортируем по расходам или по конверсии, по желанию ---
campaign_stats = campaign_stats.sort_values(by="Clicks", ascending=False)

# --- Настройки отображения ---
pd.set_option("display.max_columns", None)     # показывать все столбцы
pd.set_option("display.expand_frame_repr", False)  # не переносить строки
pd.set_option("display.max_rows", 20)          # максимум строк в выводе (можно изменить)

# --- Вывод без индекса ---
print(campaign_stats.to_string(index=False))


                   Campaign  Impressions    Spend  Clicks  Conversion Rate (%)  Cost per Click
      performancemax_eng_DE     21007536 34183.45  160175                 0.76            0.21
          youtube_shorts_DE      8481054 14149.22   57873                 0.68            0.24
               discovery_DE      6912369  9750.63   56889                 0.82            0.17
          12.07.2023wide_DE      3302301  9471.52   22768                 0.69            0.42
    1performancemax_wide_PL      2569396  2961.38   11747                 0.46            0.25
            02.07.23wide_DE       594807  6913.60   10281                 1.73            0.67
         discovery_wide1_AT      1270651  1308.18    8852                 0.70            0.15
   04.07.23recentlymoved_DE       417891  4523.31    7611                 1.82            0.59
     performancemax_wide_AT       975206  3491.74    7595                 0.78            0.46
              03.07.23women       348089  4219.75 

Визуализирую топ 10 по конверсии (из показа в клик) и по цене за клик.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Топ-10 по конверсии и по цене за клик ---
top_conversion = campaign_stats.sort_values("Conversion Rate (%)", ascending=False).head(10)
top_cpc = campaign_stats.sort_values("Cost per Click", ascending=True).head(10)

# --- Сабплоты ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Топ-10 кампаний по конверсии", "Топ-10 по минимальной цене за клик"),
    horizontal_spacing=0.18  # увеличиваем расстояние между графиками
)

# --- Левый график: Конверсия ---
fig.add_trace(
    go.Bar(
        x=top_conversion["Conversion Rate (%)"],
        y=top_conversion["Campaign"],
        orientation="h",
        marker_color="#1f77b4",
        name="Конверсия (%)",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Конверсия: %{x:.2f}%<br>"
            "Цена за клик: %{customdata:.2f} €<extra></extra>"
        ),
        customdata=top_conversion["Cost per Click"]
    ),
    row=1, col=1
)

# --- Правый график: Цена за клик ---
fig.add_trace(
    go.Bar(
        x=top_cpc["Cost per Click"],
        y=top_cpc["Campaign"],
        orientation="h",
        marker_color="#2ca02c",
        name="Цена за клик",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Цена за клик: %{x:.2f} €<br>"
            "Конверсия: %{customdata:.2f}%<extra></extra>"
        ),
        customdata=top_cpc["Conversion Rate (%)"]
    ),
    row=1, col=2
)

# --- Оформление ---
fig.update_layout(
    title="📊 Эффективность рекламных кампаний",
    showlegend=False,
    height=600,
    width=1150,
)

# чтобы «топ» был сверху
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)

fig.show()


In [ ]:
fig.write_json("top_campaigns.json")

Тут пыталась посмотреть как распределяются рекламные кампании внутри источников рекламы.

In [ ]:
# --- Двухуровневая агрегация по Source и Campaign ---
campaign_stats2 = (
    spend_df[spend_df["Campaign"].notna()]  # исключаем пустые Campaign
    .groupby(["Source", "Campaign"], dropna=False)[["Impressions", "Spend", "Clicks"]]
    .sum()
    .reset_index()
)

# --- Добавляем метрики ---
campaign_stats2["Conversion Rate (%)"] = (
    campaign_stats2["Clicks"] / campaign_stats2["Impressions"] * 100
).round(2)

campaign_stats2["Cost per Click"] = (
    campaign_stats2["Spend"] / campaign_stats2["Clicks"]
).round(2)

# --- Сортировка по расходам или по конверсии ---
campaign_stats2 = campaign_stats2.sort_values(by=["Source", "Spend"], ascending=[True, False])

# --- Настройки отображения ---
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_rows", 50)

# --- Вывод без индекса ---
print(campaign_stats2.to_string(index=False))


      Source                    Campaign  Impressions    Spend  Clicks  Conversion Rate (%)  Cost per Click
Facebook Ads             02.07.23wide_DE       594807  6913.60   10281                 1.73            0.67
Facebook Ads    04.07.23recentlymoved_DE       417891  4523.31    7611                 1.82            0.59
Facebook Ads               03.07.23women       348089  4219.75    7139                 2.05            0.59
Facebook Ads              07.07.23LAL_DE       335725  4200.37    5813                 1.73            0.72
Facebook Ads   12.09.23interests_Uxui_DE       319463  3753.06    6301                 1.97            0.60
Facebook Ads      24.09.23retargeting_DE       252080  2817.29    4143                 1.64            0.68
Facebook Ads             17.03.24wide_AT       106981  1435.24     657                 0.61            2.18
Facebook Ads                15.07.23b_DE        54268  1256.50     881                 1.62            1.43
Facebook Ads             30.

 Рекомендация: gen_analyst_DE - также протестировать как в Google Ads будут заходить обьявления по направлениям, и потом сфокусироваться на уменьшении цены. Так как тут может быть потенциал. И как тест позапускать по крупным городам и по регионам, протестировать есть ли разница.

Другой вид графика.

In [ ]:
# --- Топ-10 по конверсии ---
top_conv = campaign_stats2.sort_values("Conversion Rate (%)", ascending=False).head(10)

# --- Топ-10 по цене за клик ---
top_cpc2 = campaign_stats2.sort_values("Cost per Click", ascending=True).head(10)

# --- Сабплоты ---
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Топ-10 по конверсии", "Топ-10 по цене за клик"),
    horizontal_spacing=0.15
)

# --- График конверсии ---
fig2.add_trace(
    go.Bar(
        x=top_conv["Campaign"],
        y=top_conv["Conversion Rate (%)"],
        name="Conversion Rate",
        hovertemplate=
            "Campaign: %{x}<br>" +
            "Conversion Rate: %{y:.2f}%<br>" +
            "Cost per Click: %{customdata[0]:.2f}<br>" +
            "Source: %{customdata[1]}",
        customdata=top_conv[["Cost per Click", "Source"]].values,
        marker_color="#2E91E5"
    ),
    row=1, col=1
)

# --- График цены за клик ---
fig2.add_trace(
    go.Bar(
        x=top_cpc2["Campaign"],
        y=top_cpc2["Cost per Click"],
        name="Cost per Click",
        hovertemplate=
            "Campaign: %{x}<br>" +
            "Cost per Click: %{y:.2f}<br>" +
            "Conversion Rate: %{customdata[0]:.2f}%<br>" +
            "Source: %{customdata[1]}",
        customdata=top_cpc2[["Conversion Rate (%)", "Source"]].values,
        marker_color="#2ca02c"
    ),
    row=1, col=2
)

fig2.update_layout(
    #height=500,
    #width=1000,
    showlegend=False,
)

fig2.show()


In [ ]:
fig2.write_json("top_campaigns2.json")

Тут также пыталась посмотреть по Ad Gruop внутри Source.

In [ ]:
# --- Двухуровневая агрегация по Source и AdGroup ---
campaign_stats3 = (
    spend_df[spend_df["AdGroup"].notna()]  # исключаем пустые AdGroup
    .groupby(["Source", "AdGroup"], dropna=False)[["Impressions", "Spend", "Clicks"]]
    .sum()
    .reset_index()
)

# --- Добавляем метрики ---
campaign_stats3["Conversion Rate (%)"] = (
    campaign_stats3["Clicks"] / campaign_stats3["Impressions"] * 100
).round(2)

campaign_stats3["Cost per Click"] = (
    campaign_stats3["Spend"] / campaign_stats3["Clicks"]
).round(2)

# --- Сортировка по расходам или по конверсии ---
campaign_stats3 = campaign_stats3.sort_values(by=["Source", "Spend"], ascending=[True, False])

# --- Настройки отображения ---
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_rows", 50)

# --- Вывод без индекса ---
print(campaign_stats3.to_string(index=False))


      Source                       AdGroup  Impressions    Spend  Clicks  Conversion Rate (%)  Cost per Click
Facebook Ads                          wide       886350 10478.32   13744                 1.55            0.76
Facebook Ads                 recentlymoved       442003  4792.61    7773                 1.76            0.62
Facebook Ads                         women       394045  4577.00    7506                 1.90            0.61
Facebook Ads                          LAL1       354612  4368.00    5942                 1.68            0.74
Facebook Ads                   retargeting       252080  2817.29    4143                 1.64            0.68
Facebook Ads          interest_work_WebDev       193801  2288.53    3801                 1.96            0.60
Facebook Ads   interest_programming_WebDev       191162  2140.62    3038                 1.59            0.70
Facebook Ads                             b        54268  1256.50     881                 1.62            1.43
Facebook A

Думаю стоит посмотреть на только по кампаниям, но и по источнику вцелом.

In [6]:
# --- Агрегация по источникам ---
campaign_stats2 = (
    spend_df.groupby("Source", dropna=True)[["Impressions", "Spend", "Clicks"]]
    .sum()
    .reset_index()
)

# --- Добавляем метрики ---
campaign_stats2["Conversion Rate (%)"] = (
    campaign_stats2["Clicks"] / campaign_stats2["Impressions"] * 100
).round(2)

campaign_stats2["Cost per Click"] = (
    campaign_stats2["Spend"] / campaign_stats2["Clicks"]
).round(2)

# --- Сортируем по расходам или по конверсии, по желанию ---
campaign_stats2 = campaign_stats2.sort_values(by="Clicks", ascending=False)

# --- Настройки отображения ---
pd.set_option("display.max_columns", None)     # показывать все столбцы
pd.set_option("display.expand_frame_repr", False)  # не переносить строки
pd.set_option("display.max_rows", 20)          # максимум строк в выводе (можно изменить)

# --- Вывод без индекса ---
print(campaign_stats2.to_string(index=False))

        Source  Impressions    Spend  Clicks  Conversion Rate (%)  Cost per Click
    Google Ads     32752334 57798.60  248487                 0.76            0.23
       Organic            0     0.00   59089                  inf            0.00
   Youtube Ads      8655978 14633.33   59061                 0.68            0.25
  Facebook Ads      2850200 33754.72   48133                 1.69            0.70
    Tiktok Ads      5007212 11985.67   28268                 0.56            0.42
Telegram posts       705415  6860.36   16777                 2.38            0.41
      Bloggers       738460 13439.00   14250                 1.93            0.94
           SMM        23772  7269.52   11521                48.46            0.63
           CRM            0     0.00    7995                  inf            0.00
       Webinar       301670  2874.04    3241                 1.07            0.89
          Test        43969   608.21    1226                 2.79            0.50
   Partnership  

2. Оцените эффективность различных маркетинговых источников Source в
генерировании качественных лидов.

Думаю сразу посмотреть не только по Source, а еще и по Campaign и AdGruop, так как они могут отличаться даже в рамках одного Source. Качественными будем считать всех заинтересованных (то есть всех кто есть в Deals), но в идеале и посмотреть только успешные сделки (у кого Stage - Payment Done).

In [ ]:
deals_df = pd.read_excel("Deals_f_clean.xlsx")

display(deals_df.head())

deals_df.info()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA_hours
0,5805028000056864695,Ben Hall,NaT,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,2024-06-21 15:30:00,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaT,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Morning,2024-06-21 15:23:00,6.0,NaN,0.0,2000.0,5.805028e+18,NaN,NaN,NaN
2,5805028000056832357,Ulysses Adams,2024-06-21,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,2024-06-21 14:45:00,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN,0.4453
3,5805028000056824246,Eva Kent,2024-06-21,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,2024-06-21 13:32:00,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN,1.0011
4,5805028000056873292,Ben Hall,2024-06-21,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,2024-06-21 13:21:00,NaN,NaN,NaN,NaN,5.805028e+18,NaN,NaN,0.8867


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19822 entries, 0 to 19821
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id                   19822 non-null  int64         
 1   Deal Owner Name      19793 non-null  object        
 2   Closing Date         13136 non-null  datetime64[ns]
 3   Quality              17586 non-null  object        
 4   Stage                19822 non-null  object        
 5   Lost Reason          14353 non-null  object        
 6   Page                 19822 non-null  object        
 7   Campaign             15583 non-null  object        
 8   SLA                  14995 non-null  object        
 9   Content              13798 non-null  object        
 10  Term                 12101 non-null  object        
 11  Source               19822 non-null  object        
 12  Payment Type         483 non-null    object        
 13  Product              3537 non-n

Эффективность источников лидов уже в разрезе продаж.

In [ ]:
# --- Агрегируем данные ---
source_stats = (
    deals_df.groupby("Source", dropna=True)
    .agg(
        total_deals=("Id", "count"),
        successful_deals=("Stage", lambda x: (x == "Payment Done").sum()),
        total_sales=("Offer Total Amount", lambda x: x[deals_df.loc[x.index, "Stage"] == "Payment Done"].sum())
    )
    .reset_index()
)

# --- Добавляем коэффициент конверсии ---
source_stats["conversion_rate (%)"] = (
    source_stats["successful_deals"] / source_stats["total_deals"] * 100
).round(1)

# --- Переставим колонки в логичном порядке ---
source_stats = source_stats[[
    "Source", "total_deals", "successful_deals", "conversion_rate (%)", "total_sales"
]]

# --- Сортировка по продажам ---
source_stats = source_stats.sort_values(by="total_deals", ascending=False)

# --- Настройки вывода ---
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)  # чтобы не было переноса
pd.set_option("display.max_rows", 50)

# --- Вывод без индекса ---
print(source_stats.to_string(index=False))


        Source  total_deals  successful_deals  conversion_rate (%)  total_sales
  Facebook Ads         4730               202                  4.3    1550700.0
    Google Ads         4114               172                  4.2    1275000.0
    Tiktok Ads         2003                56                  2.8     368500.0
           SMM         1669                91                  5.5     602000.0
   Youtube Ads         1618                53                  3.3     415500.0
       Organic         1498               146                  9.7    1070001.0
           CRM         1456                24                  1.6     167500.0
      Bloggers         1074                39                  3.6     288000.0
Telegram posts          993                40                  4.0     300000.0
       Webinar          306                26                  8.5     182700.0
   Partnership          203                 4                  2.0      36000.0
          Test          156             

Добавим коэффициент ROI

In [ ]:

# --- 2️⃣ Агрегация по расходам (spend_df) ---
spend_stats = (
    spend_df.groupby("Source", dropna=True)["Spend"]
    .sum()
    .reset_index()
    .rename(columns={"Spend": "total_spend"})
)

# --- 3️⃣ Объединяем две таблицы по Source ---
merged_stats = pd.merge(source_stats, spend_stats, on="Source", how="left")

# --- 4️⃣ Добавляем метрики ---
merged_stats["conversion_rate (%)"] = (
    merged_stats["successful_deals"] / merged_stats["total_deals"] * 100
).round(1)

merged_stats["roi (sales/spend)"] = (
    merged_stats["total_sales"] / merged_stats["total_spend"]
).round(2)

# --- 5️⃣ Упорядочиваем столбцы ---
merged_stats = merged_stats[
    [
        "Source",
        "total_deals",
        "successful_deals",
        "conversion_rate (%)",
        "total_sales",
        "total_spend",
        "roi (sales/spend)"
    ]
]

# --- Сортировка по продажам ---
merged_stats = merged_stats.sort_values(by="total_sales", ascending=False)

# --- 6️⃣ Настройки вывода ---
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

# --- 7️⃣ Вывод без индекса ---
print(merged_stats.to_string(index=False))


        Source  total_deals  successful_deals  conversion_rate (%)  total_sales  total_spend  roi (sales/spend)
  Facebook Ads         4730               202                  4.3    1550700.0     33754.72              45.94
    Google Ads         4114               172                  4.2    1275000.0     57798.60              22.06
       Organic         1498               146                  9.7    1070001.0         0.00                inf
           SMM         1669                91                  5.5     602000.0      7269.52              82.81
   Youtube Ads         1618                53                  3.3     415500.0     14633.33              28.39
    Tiktok Ads         2003                56                  2.8     368500.0     11985.67              30.75
Telegram posts          993                40                  4.0     300000.0      6860.36              43.73
      Bloggers         1074                39                  3.6     288000.0     13439.00            

График - какой доход на 1 евро затрат.

In [ ]:
import plotly.express as px
import numpy as np

# --- 1️⃣ Отфильтруем корректные значения ---
roi_plot = merged_stats[
    merged_stats["roi (sales/spend)"].replace([np.inf, -np.inf], np.nan).notna()
].copy()

# --- 2️⃣ Сортируем по убыванию ROI ---
roi_plot = roi_plot.sort_values(by="roi (sales/spend)", ascending=False)

# --- 3️⃣ Строим график ---
fig = px.bar(
    roi_plot,
    x="Source",
    y="roi (sales/spend)",
    title="💰 Эффективность затрат по источникам (ROI = продажи / расходы)",
    labels={"roi (sales/spend)": "Коэффициент окупаемости (ROI)", "Source": "Источник"},
    text="roi (sales/spend)",
    color_discrete_sequence=["#2E91E5"]
)

# --- 4️⃣ Настройки внешнего вида ---
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(
    xaxis_title="Источник",
    yaxis_title="ROI (продажи / расходы)",
    uniformtext_minsize=8,
    uniformtext_mode="hide",
    bargap=0.3,
    showlegend=False,
)

fig.show()


In [ ]:
fig.write_json("source_roi.json")

Теперь по кампаниям

In [ ]:
# --- 1️⃣ Агрегируем данные по сделкам ---
deals_grouped = (
    deals_df.groupby("Campaign", dropna=True)
    .agg(
        total_deals=("Id", "count"),
        successful_deals=("Stage", lambda x: (x == "Payment Done").sum()),
        total_sales=("Offer Total Amount", lambda x: x[deals_df.loc[x.index, "Stage"] == "Payment Done"].sum())
    )
    .reset_index()
)

# --- 2️⃣ Рассчитываем конверсию ---
deals_grouped["conversion_rate (%)"] = (
    deals_grouped["successful_deals"] / deals_grouped["total_deals"] * 100
).round(2)

# --- 3️⃣ Агрегируем расходы по кампаниям ---
spend_grouped = (
    spend_df.groupby("Campaign", dropna=True)["Spend"]
    .sum()
    .reset_index()
    .rename(columns={"Spend": "total_spend"})
)

# --- 4️⃣ Объединяем таблицы по Campaign ---
merged_campaign_stats = pd.merge(
    deals_grouped,
    spend_grouped,
    on="Campaign",
    how="inner"  # только кампании, которые есть в spend_df
)

# --- 5️⃣ Рассчитываем ROI ---
merged_campaign_stats["ROI"] = (
    merged_campaign_stats["total_sales"] / merged_campaign_stats["total_spend"]
).round(2)

# --- 2️⃣ Сортируем по убыванию total_sales ---
merged_campaign_stats = merged_campaign_stats.sort_values(by="total_sales", ascending=False)


# --- 6️⃣ Настройки отображения ---
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

# --- 7️⃣ Вывод таблицы без индекса ---
print(merged_campaign_stats.to_string(index=False))


                   Campaign  total_deals  successful_deals  total_sales  conversion_rate (%)  total_spend    ROI
          youtube_shorts_DE         1596                53     415500.0                 3.32     14149.22  29.37
            02.07.23wide_DE          943                52     394000.0                 5.51      6913.60  56.99
          12.07.2023wide_DE         1531                48     333500.0                 3.14      9471.52  35.21
              03.07.23women          595                31     253000.0                 5.21      4219.75  59.96
             07.07.23LAL_DE          529                28     243000.0                 5.29      4200.37  57.85
  12.09.23interests_Uxui_DE          515                27     229000.0                 5.24      3753.06  61.02
   04.07.23recentlymoved_DE          734                31     187200.0                 4.22      4523.31  41.39
     24.09.23retargeting_DE          472                17     143000.0                 3.60    

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1️⃣ Топ-10 кампаний по продажам ---
top10_sales = merged_campaign_stats.sort_values(by="total_sales", ascending=False).head(10)
top10_sales = top10_sales.sort_values(by="total_sales", ascending=True)


fig1 = go.Figure()

fig1.add_trace(go.Bar(
    y=top10_sales["Campaign"],  # <-- заменяем x → y
    x=top10_sales["total_sales"],  # <-- заменяем y → x
    orientation='h',  # <-- добавляем ориентацию "горизонтально"
    text=[f"{v:,.0f}" for v in top10_sales["total_sales"]],
    textposition="outside",
    marker_color="#2E91E5",
    hovertemplate=(
        "<b>%{y}</b><br>"  # <-- здесь тоже y вместо x
        "💰 Продажи: %{x:,.0f}<br>"
        "🔹 Конверсия: %{customdata[0]:.2f}%<br>"
        "📈 ROI: %{customdata[1]:.2f}<extra></extra>"
    ),
    customdata=top10_sales[["conversion_rate (%)", "ROI"]].values
))

fig1.update_layout(
    title="🏆 Топ-10 кампаний по сумме продаж",
    yaxis_title="Кампания",
    xaxis_title="Сумма продаж (€)",
    uniformtext_minsize=10,
    uniformtext_mode="hide",
    bargap=0.3,
    height=600
)

fig1.show()


# --- 2️⃣ Сапплоты: кампании с потенциалом ---
top5_conv = merged_campaign_stats.sort_values(by="conversion_rate (%)", ascending=False).head(5)
top5_roi = merged_campaign_stats[merged_campaign_stats["ROI"].notna()].sort_values(by="ROI", ascending=False).head(5)

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("🔥 Топ-5 по конверсии", "💸 Топ-5 по ROI"),
    horizontal_spacing=0.15
)

# --- Левый график (конверсия) ---
fig2.add_trace(
    go.Bar(
        x=top5_conv["Campaign"],
        y=top5_conv["conversion_rate (%)"],
        text=[f"{v:.2f}%" for v in top5_conv["conversion_rate (%)"]],
        textposition="outside",
        marker_color="#00C851",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Конверсия: %{y:.2f}%<br>"
            "ROI: %{customdata[0]:.2f}<br>"
            "Продажи: %{customdata[1]:,.0f}<extra></extra>"
        ),
        customdata=top5_conv[["ROI", "total_sales"]].values
    ),
    row=1, col=1
)

# --- Правый график (ROI) ---
fig2.add_trace(
    go.Bar(
        x=top5_roi["Campaign"],
        y=top5_roi["ROI"],
        text=[f"{v:.2f}" for v in top5_roi["ROI"]],
        textposition="outside",
        marker_color="#FF8800",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "ROI: %{y:.2f}<br>"
            "Конверсия: %{customdata[0]:.2f}%<br>"
            "Продажи: %{customdata[1]:,.0f}<extra></extra>"
        ),
        customdata=top5_roi[["conversion_rate (%)", "total_sales"]].values
    ),
    row=1, col=2
)

fig2.update_layout(
    height=500,
    title_text="🚀 Кампании с наибольшим потенциалом",
    showlegend=False,
    bargap=0.4,
)

fig2.update_yaxes(title_text="Конверсия (%)", row=1, col=1)
fig2.update_yaxes(title_text="ROI", row=1, col=2)

fig2.show()


In [ ]:
fig1.write_json("top10_campaigns.json")
fig2.write_json("top5_potential.json")

теперь по AdGroup

In [ ]:
# --- 1️⃣ Агрегируем данные по сделкам ---
deals_grouped2 = (
    deals_df.groupby("Term", dropna=True)
    .agg(
        total_deals=("Id", "count"),
        successful_deals=("Stage", lambda x: (x == "Payment Done").sum()),
        total_sales=("Offer Total Amount", lambda x: x[deals_df.loc[x.index, "Stage"] == "Payment Done"].sum())
    )
    .reset_index()
)

# --- 2️⃣ Рассчитываем конверсию ---
deals_grouped2["conversion_rate (%)"] = (
    deals_grouped2["successful_deals"] / deals_grouped2["total_deals"] * 100
).round(2)

# --- 3️⃣ Агрегируем расходы по кампаниям ---
spend_grouped2 = (
    spend_df.groupby("AdGroup", dropna=True)["Spend"]
    .sum()
    .reset_index()
    .rename(columns={"Spend": "total_spend"})
)

# --- 4️⃣ Объединяем таблицы по Campaign ---
merged_campaign_stats2 = pd.merge(
    deals_grouped2,
    spend_grouped2,
    left_on="Term",     # столбец из deals_grouped2
    right_on="AdGroup", # столбец из spend_grouped2
    how="inner"         # только совпадающие значения
)

merged_campaign_stats2 = merged_campaign_stats2.drop(columns=["AdGroup"])


# --- 5️⃣ Рассчитываем ROI ---
merged_campaign_stats2["ROI"] = (
    merged_campaign_stats2["total_sales"] / merged_campaign_stats2["total_spend"]
).round(2)

# --- 2️⃣ Сортируем по убыванию total_sales ---
merged_campaign_stats2 = merged_campaign_stats2.sort_values(by="total_sales", ascending=False)


# --- 6️⃣ Настройки отображения ---
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

# --- 7️⃣ Вывод таблицы без индекса ---
print(merged_campaign_stats2.to_string(index=False))


                       Term  total_deals  successful_deals  total_sales  conversion_rate (%)  total_spend   ROI
                       wide         3583               118     843500.0                 3.29     24296.22 34.72
                 Com_august         1495                51     410000.0                 3.41     12723.39 32.22
                      women          626                31     253000.0                 4.95      4577.00 55.28
                       LAL1          535                28     243000.0                 5.23      4368.00 55.63
              recentlymoved          741                31     187200.0                 4.18      4792.61 39.06
       interest_work_WebDev          301                15     143500.0                 4.98      2288.53 62.70
                retargeting          472                17     143000.0                 3.60      2817.29 50.76
interest_programming_WebDev          256                12      85500.0                 4.69      2140.6

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1️⃣ Топ-10 кампаний по продажам ---
top11_sales = merged_campaign_stats2.sort_values(by="total_sales", ascending=False).head(11)
top11_sales = top11_sales.sort_values(by="total_sales", ascending=True)


fig3 = go.Figure()

fig3.add_trace(go.Bar(
    y=top11_sales["Term"],  # <-- заменяем x → y
    x=top11_sales["total_sales"],  # <-- заменяем y → x
    orientation='h',  # <-- добавляем ориентацию "горизонтально"
    text=[f"{v:,.0f}" for v in top10_sales["total_sales"]],
    textposition="outside",
    marker_color="#2E91E5",
    hovertemplate=(
        "<b>%{y}</b><br>"  # <-- здесь тоже y вместо x
        "💰 Продажи: %{x:,.0f}<br>"
        "🔹 Конверсия: %{customdata[0]:.2f}%<br>"
        "📈 ROI: %{customdata[1]:.2f}<extra></extra>"
    ),
    customdata=top10_sales[["conversion_rate (%)", "ROI"]].values
))

fig3.update_layout(
    title="🏆 Прибыльные группы объявлений",
    yaxis_title="Группа обьявлений",
    xaxis_title="Сумма продаж (€)",
    uniformtext_minsize=10,
    uniformtext_mode="hide",
    bargap=0.3,
    height=600
)

fig3.show()




In [ ]:
fig3.write_json("top11_adgroup.json")

Еще было бы неплохо проанализировать эффективность конкретных обьявлений.
Но в целом мы должны ориентироваться по сумме продаж и roi как определению кампаний/групп обьявлений с потенциалом.
Что тут не учтено - сколько времени было запущено обьявление, если например недавно и много сделок в промежуточной стадии, тогда сравнение с другими группами обьявлений/кампаниями будет неправильным.
Также не учитывается оплата труда, если к примеру 120 сделок по одной компании принося выручку также как и 18 другой, то скорее всего затрат трудо-часов на 18 идет меньше и их можно считать более выгодными. В целом тут еще много можно проанализировать. И я бы сделки разделила бы по когортам для сравнения.

Еще хочу сделать воронки по источникам, чтоб наглядно посмотреть на их эффективность.

In [7]:
import pandas as pd

spend_df = pd.read_excel("Spend_f_clean.xlsx")
deals_df = pd.read_excel("Deals_f_clean.xlsx", dtype={"Id": str,"Contact Name": str})

Пока что таблицей.

In [15]:
import pandas as pd

# --- Агрегация данных из spend_df ---
spend_agg = spend_df.groupby("Source", dropna=True).agg(
    Impressions=("Impressions", "sum"),
    Clicks=("Clicks", "sum"),
    Spend=("Spend", "sum")
).reset_index()

# --- Агрегация данных из deals_df ---
deals_agg = deals_df.groupby("Source", dropna=True).agg(
    Deals=("Id", "count"),  # общее количество сделок
    Successful_Deals=("Stage", lambda x: (x=="Payment Done").sum()),  # успешные сделки
    Revenue=("Offer Total Amount", lambda x: x[deals_df.loc[x.index, "Stage"]=="Payment Done"].sum())  # сумма успешных
).reset_index()

# --- Объединяем таблицы ---
summary_df = pd.merge(spend_agg, deals_agg, on="Source", how="outer")

# --- 2️⃣ Сортируем по убыванию Revenue ---
summary_df = summary_df.sort_values(by="Revenue", ascending=False)


# --- Настройки отображения ---
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

# --- Вывод без индекса ---
print(summary_df.to_string(index=False))


        Source  Impressions  Clicks    Spend  Deals  Successful_Deals   Revenue
  Facebook Ads      2850200   48133 33754.72 4730.0             202.0 1550700.0
    Google Ads     32752334  248487 57798.60 4114.0             172.0 1275000.0
       Organic            0   59089     0.00 1498.0             146.0 1070001.0
           SMM        23772   11521  7269.52 1669.0              91.0  602000.0
   Youtube Ads      8655978   59061 14633.33 1618.0              53.0  415500.0
    Tiktok Ads      5007212   28268 11985.67 2003.0              56.0  368500.0
Telegram posts       705415   16777  6860.36  993.0              40.0  300000.0
      Bloggers       738460   14250 13439.00 1074.0              39.0  288000.0
       Webinar       301670    3241  2874.04  306.0              26.0  182700.0
           CRM            0    7995     0.00 1456.0              24.0  167500.0
   Partnership            0     350     0.00  203.0               4.0   36000.0
          Test        43969    1226   60

In [16]:
import plotly.graph_objects as go

# Пример для одного источника (берем из summary_df)
source_row = summary_df.iloc[0]

stages = ["Impressions", "Clicks", "Deals", "Successful Deals"]
values = [
    source_row["Impressions"],
    source_row["Clicks"],
    source_row["Deals"],
    source_row["Successful_Deals"]
]

# Вычисляем конверсии относительно предыдущей стадии
conversions = [
    values[1]/values[0] if values[0] else 0,
    values[2]/values[1] if values[1] else 0,
    values[3]/values[2] if values[2] else 0
]

fig = go.Figure(go.Funnel(
    y=stages,
    x=values,
    textinfo="value+percent previous",  # показывать % относительно предыдущей стадии
    marker={"color": ["#2E91E5", "#E15F99", "#1CA71C", "#FB0D0D"]}
))

fig.update_layout(title=f"Воронка конверсий для источника {source_row['Source']}")
fig.show()


In [20]:
# Создаем текстовое представление стадий с конверсией
summary_df["Funnel"] = (
    summary_df["Impressions"].astype(str) +
    " - " + summary_df["Conv_Clicks"].astype(str) + "% → " +
    summary_df["Clicks"].astype(str) +
    " - " + summary_df["Conv_Deals"].astype(str) + "% → " +
    summary_df["Deals"].astype(str) +
    " - " + summary_df["Conv_Success"].astype(str) + "% → " +
    summary_df["Successful_Deals"].astype(str)
)

# Выводим только Source и конвейер
table_df = summary_df[["Source", "Funnel"]]
print(table_df.to_string(index=False))


        Source                                                  Funnel
  Facebook Ads   2850200 - 1.7% → 48133 - 9.8% → 4730.0 - 4.3% → 202.0
    Google Ads 32752334 - 0.8% → 248487 - 1.7% → 4114.0 - 4.2% → 172.0
       Organic         0 - inf% → 59089 - 2.5% → 1498.0 - 9.7% → 146.0
           SMM    23772 - 48.5% → 11521 - 14.5% → 1669.0 - 5.5% → 91.0
   Youtube Ads    8655978 - 0.7% → 59061 - 2.7% → 1618.0 - 3.3% → 53.0
    Tiktok Ads    5007212 - 0.6% → 28268 - 7.1% → 2003.0 - 2.8% → 56.0
Telegram posts      705415 - 2.4% → 16777 - 5.9% → 993.0 - 4.0% → 40.0
      Bloggers     738460 - 1.9% → 14250 - 7.5% → 1074.0 - 3.6% → 39.0
       Webinar       301670 - 1.1% → 3241 - 9.4% → 306.0 - 8.5% → 26.0
           CRM          0 - inf% → 7995 - 18.2% → 1456.0 - 1.6% → 24.0
   Partnership             0 - inf% → 350 - 58.0% → 203.0 - 2.0% → 4.0
          Test        43969 - 2.8% → 1226 - 12.7% → 156.0 - 1.9% → 3.0
       Offline                 0 - inf% → 57 - 3.5% → 2.0 - 0.0% → 0.0
      

In [21]:
import plotly.graph_objects as go

# Фильтруем только те источники, где есть доход
summary_filtered = summary_df[summary_df["Revenue"].notna() & (summary_df["Revenue"] > 0)].copy()

# Рассчитаем конверсии
summary_filtered["Conv_Clicks"] = (summary_filtered["Clicks"] / summary_filtered["Impressions"] * 100).round(1)
summary_filtered["Conv_Deals"] = (summary_filtered["Deals"] / summary_filtered["Clicks"] * 100).round(1)
summary_filtered["Conv_Success"] = (summary_filtered["Successful_Deals"] / summary_filtered["Deals"] * 100).round(1)

# Формируем колонки с подписями под значениями
impressions_col = summary_filtered["Impressions"].astype(str) + "<br>(" + summary_filtered["Conv_Clicks"].astype(str) + "%)"
clicks_col = summary_filtered["Clicks"].astype(str) + "<br>(" + summary_filtered["Conv_Deals"].astype(str) + "%)"
deals_col = summary_filtered["Deals"].astype(str) + "<br>(" + summary_filtered["Conv_Success"].astype(str) + "%)"
success_col = summary_filtered["Successful_Deals"].astype(str)

# Создаем таблицу
fig = go.Figure(data=[go.Table(
    header=dict(values=["Source", "Impressions → Clicks → Deals → Successful Deals"],
                fill_color='paleturquoise',
                align='center'),
    cells=dict(values=[
        summary_filtered["Source"],
        impressions_col + " → " + clicks_col + " → " + deals_col + " → " + success_col
    ],
    fill_color='lavender',
    align='center'))
])

fig.show()


In [22]:
import plotly.graph_objects as go

# Фильтруем только источники с доходом
summary_filtered = summary_df[summary_df["Revenue"].notna() & (summary_df["Revenue"] > 0)].copy()

# Рассчитаем конверсии
summary_filtered["Conv_Clicks"] = (summary_filtered["Clicks"] / summary_filtered["Impressions"] * 100).round(1)
summary_filtered["Conv_Deals"] = (summary_filtered["Deals"] / summary_filtered["Clicks"] * 100).round(1)
summary_filtered["Conv_Success"] = (summary_filtered["Successful_Deals"] / summary_filtered["Deals"] * 100).round(1)

# Формируем строки с числами и конверсиями под ними
impressions_col = summary_filtered["Impressions"].astype(str) + "<br>(" + summary_filtered["Conv_Clicks"].astype(str) + "%)"
clicks_col = summary_filtered["Clicks"].astype(str) + "<br>(" + summary_filtered["Conv_Deals"].astype(str) + "%)"
deals_col = summary_filtered["Deals"].astype(str) + "<br>(" + summary_filtered["Conv_Success"].astype(str) + "%)"
success_col = summary_filtered["Successful_Deals"].astype(str)  # можно добавить конверсию если нужно

# Создаем таблицу
fig = go.Figure(data=[go.Table(
    header=dict(values=["Source", "Impressions", "Clicks", "Deals", "Successful Deals"],
                fill_color='paleturquoise',
                align='center'),
    cells=dict(values=[
        summary_filtered["Source"],
        impressions_col,
        clicks_col,
        deals_col,
        success_col
    ],
    fill_color='lavender',
    align='center'))
])

fig.show()


In [23]:
import plotly.graph_objects as go

# Фильтруем только источники с доходом
summary_filtered = summary_df[summary_df["Revenue"].notna() & (summary_df["Revenue"] > 0)].copy()

# Рассчитаем конверсии
summary_filtered["Conv_Clicks"] = (summary_filtered["Clicks"] / summary_filtered["Impressions"] * 100).round(1)
summary_filtered["Conv_Deals"] = (summary_filtered["Deals"] / summary_filtered["Clicks"] * 100).round(1)
summary_filtered["Conv_Success"] = (summary_filtered["Successful_Deals"] / summary_filtered["Deals"] * 100).round(1)

# Формируем строки с числами
impressions_col = summary_filtered["Impressions"].astype(str)
clicks_col = summary_filtered["Clicks"].astype(str)
deals_col = summary_filtered["Deals"].astype(str)
success_col = summary_filtered["Successful_Deals"].astype(str)

# Формируем строки с конверсиями, сдвинутыми на одну вправо
conv_impressions = [""] * len(summary_filtered)  # нет конверсии перед первой стадией
conv_clicks = summary_filtered["Conv_Clicks"].astype(str) + "%"  # сдвинута к следующей стадии
conv_deals = summary_filtered["Conv_Deals"].astype(str) + "%"
conv_success = summary_filtered["Conv_Success"].astype(str) + "%"

# Создаем таблицу
fig = go.Figure(data=[go.Table(
    header=dict(values=["Source", "Impressions", "Clicks", "Deals", "Successful Deals"],
                fill_color='paleturquoise',
                align='center'),
    cells=dict(values=[
        summary_filtered["Source"],
        [f"{impr}<br>{conv}" for impr, conv in zip(impressions_col, conv_impressions)],
        [f"{clk}<br>{conv}" for clk, conv in zip(clicks_col, conv_clicks)],
        [f"{dl}<br>{conv}" for dl, conv in zip(deals_col, conv_deals)],
        [f"{succ}<br>{conv}" for succ, conv in zip(success_col, conv_success)]
    ],
    fill_color='lavender',
    align='center'))
])

fig.show()


In [24]:
fig.write_json("source_funel.json")